In [ ]:
!pip install "scikit-learn>=1.1" "POT>=0.9.3" numpy
!pip install tslearn

In [ ]:

# ===================== Import Libraries =====================
import os
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import neighbors
from sklearn.metrics import accuracy_score, average_precision_score
try:
    import scipy.sparse as sp
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False

# ---------- Custom utility: load UCR dataset (.ts format) ----------
from tslearn.utils import to_time_series_dataset

def load_ucr_dataset_tsl(data_dir, dataset_name):
    """
    Loads train and test data using tslearn's UCR/UEA loader.
    data_dir hiện không sử dụng nhưng giữ lại cho tương thích.
    """
    from tslearn.datasets import UCR_UEA_datasets
    X_train, y_train, X_test, y_test = UCR_UEA_datasets().load_dataset(dataset_name)

    print("Successfully loaded dataset:", dataset_name)
    print("Size of train data:", len(y_train))
    print("Size of test data:", len(y_test))

    return X_train, y_train, X_test, y_test

def compute_map_knn_precomputed(X_computed, X_test_computed,
                                y_train, y_test, k=1):
    """
    Tính MAP cho k-NN (metric='precomputed') theo mô tả trong paper:

      - Fit k-NN với số láng giềng k.
      - predict_proba trên test -> score cho từng lớp.
      - Với mỗi lớp c:
          AP_c = average_precision_score(1_{y_test==c}, score_c)
      - MAP = trung bình AP_c trên tất cả các lớp.

    Trả về MAP dạng phần trăm [%].
    """
    clf = neighbors.KNeighborsClassifier(
        n_neighbors=k,
        metric="precomputed",
        weights="uniform",
    )
    clf.fit(X_computed, y_train)

    proba = clf.predict_proba(X_test_computed)   # shape (n_test, n_classes)
    classes = clf.classes_

    aps = []
    for c_idx, c in enumerate(classes):
        y_true_c = (y_test == c).astype(int)
        if np.sum(y_true_c) == 0:
            # không có sample lớp c trong test -> bỏ qua
            continue

        y_score_c = proba[:, c_idx]

        # Nếu mọi score y hệt nhau thì PR curve thoái hoá, coi AP = 0
        if np.all(y_score_c == y_score_c[0]):
            aps.append(0.0)
        else:
            ap_c = average_precision_score(y_true_c, y_score_c)
            aps.append(ap_c)

    if len(aps) == 0:
        return 0.0

    return float(np.mean(aps) * 100.0)



In [ ]:
# otsw_api.py  — RAGGED-FRIENDLY (TAMLE ONLY + FARTHest-Point-Clustering SPLIT)
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Sequence, Union

# (tuỳ chọn) dùng SciPy để tăng tốc SpMM; nếu không có vẫn chạy được
try:
    import scipy.sparse as sp
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

BIG = 1e12

# =========================
# 0) HELPERS (ragged / dense)
# =========================
def _as_ragged_list(M: Union[np.ndarray, Sequence[np.ndarray]]) -> Tuple[List[np.ndarray], int]:
    """
    Chuẩn hoá đầu vào về list các mảng (n_i, d).
    Trả về (list_seq, d)
    """
    if isinstance(M, np.ndarray):
        if M.ndim != 3:
            raise ValueError("If ndarray, expect shape (m, n, d).")
        m, n, d = M.shape
        seqs = [M[i] for i in range(m)]
        return seqs, d

    seqs: List[np.ndarray] = []
    d = None
    for i, xi in enumerate(M):
        xi = np.asarray(xi, dtype=float)
        if xi.ndim != 2:
            raise ValueError(f"Sequence {i} must have shape (n_i, d).")
        if d is None:
            d = xi.shape[1]
        elif xi.shape[1] != d:
            raise ValueError("All sequences must have the same feature dimension d.")
        seqs.append(xi)
    if d is None:
        raise ValueError("Empty sequence list.")
    return seqs, d


def _linearize_points_ragged(M: Union[np.ndarray, Sequence[np.ndarray]]):
    """
    Hỗ trợ ragged:
      - P: (N, d) các điểm ghép lại
      - Sidx: (N,) id chuỗi
      - Tpos: (N,) thời gian chuẩn hoá trong [0,1) cho mỗi điểm (i / n_i)
      - lengths: (m_seq,) độ dài từng chuỗi
      - d: số kênh
    """
    seqs, d = _as_ragged_list(M)
    m_seq = len(seqs)
    lengths = np.array([xi.shape[0] for xi in seqs], dtype=int)

    P = np.vstack(seqs) if m_seq > 0 else np.zeros((0, d), dtype=float)
    Sidx = np.repeat(np.arange(m_seq, dtype=int), lengths)

    # vị trí thời gian chuẩn hoá (không dùng endpoint=1 để tránh trùng 1.0)
    Tpos_list = [(np.arange(n_i, dtype=float) / max(n_i, 1)) for n_i in lengths]
    Tpos = np.concatenate(Tpos_list) if m_seq > 0 else np.zeros((0,), dtype=float)

    return P, Sidx, Tpos, m_seq, lengths, d


def _pairwise_sqdist(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Khoảng cách 'lai' (d_tau dùng trong clustering):
      Euclid^2 trên các đặc trưng (trừ cột cuối)  +  |orderA - orderB|   (cột cuối)
    Ghi chú: ta dùng dạng "sqdist + penalty" (không sqrt) vì argmin/argmax không đổi nếu sqrt.
    """
    assert A.ndim == 2 and B.ndim == 2, "Expect 2D arrays"
    assert A.shape[1] == B.shape[1], "Dim mismatch"
    D = A.shape[1]
    if D == 0:
        return np.zeros((A.shape[0], B.shape[0]), dtype=float)
    if D == 1:
        a_ord = A[:, 0]
        b_ord = B[:, 0]
        return np.abs(a_ord[:, None] - b_ord[None, :])

    Af = A[:, :-1]
    Bf = B[:, :-1]
    aa = (Af * Af).sum(1)[:, None]
    bb = (Bf * Bf).sum(1)[None, :]
    Dsq = np.clip(aa + bb - 2.0 * (Af @ Bf.T), 0.0, None)

    a_ord = A[:, -1]
    b_ord = B[:, -1]
    Pen = np.abs(a_ord[:, None] - b_ord[None, :])
    return Dsq + Pen


# =========================
# 0.1) FARTHest-Point-Clustering (as in your LaTeX)
# =========================
def _farthest_point_clustering_centers(
    Z: np.ndarray,
    k: int,
    seed: int = 0,
) -> np.ndarray:
    """
    Chỉ thực hiện bước "select cluster centers" của Farthest Point Clustering:
      y1 <- random in Z
      for c=2..k:
        y_c <- argmax_{z in Z} min_{y in C} d_tau(z,y)

    Trả về: indices của k centers (theo index local trong Z).
    """
    n = Z.shape[0]
    if n == 0:
        raise ValueError("Empty point set.")
    k_eff = int(min(max(1, k), n))
    rng = np.random.default_rng(seed)

    # y1: random element
    i0 = int(rng.integers(0, n))
    centers = [i0]

    # dmin[z] = min distance from z to current centers
    # init with distances to first center
    dmin = _pairwise_sqdist(Z, Z[i0:i0+1]).reshape(-1)

    # iteratively add farthest point
    for _ in range(1, k_eff):
        # y_c = argmax_z dmin[z]
        ic = int(np.argmax(dmin))
        centers.append(ic)

        # update dmin = min(dmin, d(z, new_center))
        dnew = _pairwise_sqdist(Z, Z[ic:ic+1]).reshape(-1)
        dmin = np.minimum(dmin, dnew)

    return np.array(centers, dtype=int)


# =========================
# 0.2) DATA STRUCTS
# =========================
@dataclass
class _Node:
    idx: np.ndarray
    height: float
    children: List[int]          # k children node-ids (empty if leaf)
    parent: Optional[int]
    is_leaf: bool


@dataclass
class OTSWModel:
    # shared runtime fields
    P: np.ndarray                  # (N, d_aug)
    Sidx: np.ndarray               # (N,) id chuỗi
    Tpos: np.ndarray               # (N,) thời gian chuẩn hoá (0..1)
    lengths: np.ndarray            # (m_seq,)
    m_seq: int
    d: int                         # số kênh gốc (chưa augment)
    nodes: List[_Node]
    leaf_ids: List[int]
    leaf_index_map: Dict[int, int]
    edges: List[Tuple[int, int, float]]     # (parent, child, w_e)
    S_edge_leaf: object                     # (E, L) dense hoặc sp.csr_matrix
    centroids: np.ndarray                   # (num_nodes, d_aug)
    # meta
    mode: str = "tamle"
    lam_time: float = 0.0
    # caches
    point_leaf: Optional[np.ndarray] = None  # (N,)
    H: Optional[np.ndarray] = None           # (L, m_seq)
    M: Optional[np.ndarray] = None           # (E, m_seq)
    w: Optional[np.ndarray] = None           # (E,)


# =========================
# 1) ROUTING & PRECOMPUTE
# =========================
def _route_all_points_vectorized(model: OTSWModel) -> np.ndarray:
    """
    Route mọi điểm xuống lá bằng cách so khoảng cách Euclid tới centroid của các con (k-ary).
    (Lưu ý: dùng Euclid cho routing; d_tau đã dùng khi split/cluster centers.)
    """
    N = model.P.shape[0]
    leaf_of_point = np.empty(N, dtype=np.int32)
    stack = [(0, np.arange(N, dtype=np.int32))]

    nodes = model.nodes
    C = model.centroids
    P = model.P

    while stack:
        nid, idxs = stack.pop()
        nd = nodes[nid]
        if nd.is_leaf:
            j = model.leaf_index_map[nid]
            leaf_of_point[idxs] = j
            continue

        # k-ary routing: tìm child có centroid gần nhất
        children = nd.children
        X = P[idxs]
        dists = np.stack([np.linalg.norm(X - C[c], axis=1) for c in children], axis=1)  # (n, k)
        assignments = np.argmin(dists, axis=1)  # (n,)
        for ci, child_nid in enumerate(children):
            mask = (assignments == ci)
            if mask.any():
                stack.append((child_nid, idxs[mask]))

    return leaf_of_point


def _precompute_H_M(model: OTSWModel):
    """
    - point_leaf: mỗi điểm -> leaf id (0..L-1)
    - H: histogram leaf per sequence (L, m_seq), chuẩn hoá theo tổng điểm mỗi chuỗi
    - S_edge_leaf: (E, L) incidence subtree(edge) vs leaves
    - M = S @ H (E, m_seq)
    - w: edge weights
    """
    m_seq = model.m_seq
    L = len(model.leaf_ids)
    E = len(model.edges)

    # 1) route tất cả điểm -> lá
    point_leaf = _route_all_points_vectorized(model)  # (N,)
    model.point_leaf = point_leaf

    # 2) H (L, m_seq)
    H = np.zeros((L, m_seq), dtype=np.float32)
    for s in range(m_seq):
        mask = (model.Sidx == s)
        if not np.any(mask):
            continue
        counts = np.bincount(point_leaf[mask], minlength=L).astype(np.float32)
        tot = float(counts.sum())
        if tot > 0:
            counts /= tot
        H[:, s] = counts
    model.H = H

    # 3) S_edge_leaf -> CSR (nếu có SciPy) và M = S @ H
    if _HAS_SCIPY:
        SpS = sp.csr_matrix(model.S_edge_leaf)
        model.S_edge_leaf = SpS
        M = (SpS @ H).astype(np.float32)  # (E, m_seq)
    else:
        M = (model.S_edge_leaf @ H).astype(np.float32)
    model.M = M

    # 4) Trọng số cạnh
    model.w = np.array([we for _, _, we in model.edges], dtype=np.float32)


# =========================
# 2) OTSW — TAM LE (ragged OK) + Farthest Point Clustering split
# =========================
def _augment_points(seq: np.ndarray, lam_time: float) -> np.ndarray:
    n = seq.shape[0]
    t = (np.arange(n, dtype=float) / max(n, 1))[:, None] * np.sqrt(lam_time)
    return np.hstack([seq, t])


def build_otsw_tamle(
    M: Union[np.ndarray, Sequence[np.ndarray]],
    lam_time: float = 5.0,
    leaf_size: int = 16,
    max_depth: int = 20,
    seed: int = 0,
    k_split: int = 2,
) -> OTSWModel:
    """
    Xây 1 cây global theo TamLe (augment theo thời gian chuẩn hoá → ragged friendly),
    và thay thuật toán chọn tâm/split bằng Farthest Point Clustering như pseudo-code LaTeX.

    Split strategy:
      - dùng FPC để lấy k_split centers trên Xsub (theo d_tau = _pairwise_sqdist)
      - gán label theo nearest-center (argmin d_tau)
      - tạo 1 child cho mỗi cluster không rỗng → k-ary tree
    """
    P_raw, Sidx, Tpos, m_seq, lengths, d = _linearize_points_ragged(M)

    # augment từng chuỗi rồi ghép
    P_aug_list = []
    start = 0
    for s in range(m_seq):
        n_i = lengths[s]
        seq = P_raw[start:start + n_i]
        P_aug_list.append(_augment_points(seq, lam_time))
        start += n_i
    P_aug = np.vstack(P_aug_list) if P_aug_list else np.zeros((0, d + 1), dtype=float)

    nodes: List[_Node] = []
    leaf_ids: List[int] = []

    def _euclid_radius(X: np.ndarray) -> float:
        """
        Bán kính xấp xỉ để làm height: 0.5 * max distance tới 1 điểm farthest (heuristic)
        """
        if X.shape[0] <= 1:
            return 0.0
        if X.shape[0] > 1024:
            I = np.random.default_rng(0).choice(X.shape[0], 1024, replace=False)
            Y = X[I]
        else:
            Y = X
        j0 = 0
        d0 = np.linalg.norm(Y - Y[j0], axis=1)
        j1 = int(np.argmax(d0))
        d1 = np.linalg.norm(Y - Y[j1], axis=1)
        return 0.5 * float(d1.max())

    def build(idx: np.ndarray, depth: int, parent: Optional[int], seed_: int) -> int:
        Xsub = P_aug[idx]
        h = _euclid_radius(Xsub)

        nid = len(nodes)
        nodes.append(_Node(idx=idx, height=h, children=[], parent=parent, is_leaf=False))

        # leaf condition
        if idx.size <= leaf_size or depth >= max_depth or h == 0.0:
            nodes[nid].is_leaf = True
            leaf_ids.append(nid)
            return nid

        # Farthest Point Clustering centers (k_effective auto-clip)
        k_effective = int(min(max(2, k_split), idx.size))
        if k_effective < 2:
            nodes[nid].is_leaf = True
            leaf_ids.append(nid)
            return nid

        # centers on local subset Xsub
        C_local = _farthest_point_clustering_centers(Xsub, k=k_effective, seed=seed_)
        centers = Xsub[C_local]  # (k_effective, d_aug)

        # assign points to nearest center via d_tau
        lab = np.argmin(_pairwise_sqdist(Xsub, centers), axis=1)

        # k-ary split: tạo nhóm cho mỗi cluster không rỗng
        child_groups = []
        for c in range(k_effective):
            mask_c = (lab == c)
            if mask_c.any():
                child_groups.append(idx[mask_c])

        # fallback nếu chỉ ra 1 nhóm (tất cả cùng cluster) → chia đều
        if len(child_groups) < 2:
            chunk = max(1, idx.size // k_effective)
            child_groups = []
            for i in range(0, idx.size, chunk):
                child_groups.append(idx[i:i + chunk])
            if len(child_groups) < 2:
                nodes[nid].is_leaf = True
                leaf_ids.append(nid)
                return nid

        # build children
        children_nids = []
        for i, grp in enumerate(child_groups):
            child_nid = build(grp, depth + 1, nid, seed_ + i + 1)
            children_nids.append(child_nid)
        nodes[nid].children = children_nids
        return nid

    if P_aug.shape[0] == 0:
        # model rỗng
        model = OTSWModel(
            P=P_aug, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
            nodes=[_Node(idx=np.array([], dtype=int), height=0.0, children=[], parent=None, is_leaf=True)],
            leaf_ids=[0], leaf_index_map={0: 0},
            edges=[], S_edge_leaf=np.zeros((0, 1), dtype=np.float32),
            centroids=np.zeros((1, d + 1), dtype=float),
            mode="tamle", lam_time=lam_time
        )
        _precompute_H_M(model)
        return model

    # build tree
    _ = build(np.arange(P_aug.shape[0], dtype=int), 0, None, seed)

    # edges & weights
    edges: List[Tuple[int, int, float]] = []
    for cid, nd in enumerate(nodes):
        if nd.parent is not None:
            p = nodes[nd.parent]
            w = max(0.0, p.height - nd.height)
            edges.append((nd.parent, cid, w))

    # leaf mapping
    leaf_index_map = {nid: i for i, nid in enumerate(leaf_ids)}
    E, Lcnt = len(edges), len(leaf_ids)

    # build S_edge_leaf (E, L)
    S_edge_leaf = np.zeros((E, Lcnt), dtype=np.float32)

    def collect_leaves(nid_: int, out: List[int]):
        nd_ = nodes[nid_]
        if nd_.is_leaf:
            out.append(nid_)
            return
        for ch in nd_.children:
            collect_leaves(ch, out)

    for e, (pid, cid, _) in enumerate(edges):
        leaves: List[int] = []
        collect_leaves(cid, leaves)
        for ln in leaves:
            j = leaf_index_map[ln]
            S_edge_leaf[e, j] = 1.0

    # centroids for routing
    centroids = np.vstack([P_aug[nd.idx].mean(axis=0) if nd.idx.size else np.zeros((P_aug.shape[1],), dtype=float)
                           for nd in nodes])

    model = OTSWModel(
        P=P_aug, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
        nodes=nodes, leaf_ids=leaf_ids, leaf_index_map=leaf_index_map,
        edges=edges, S_edge_leaf=S_edge_leaf, centroids=centroids,
        mode="tamle", lam_time=lam_time
    )
    _precompute_H_M(model)
    return model


# =========================
# 3) DISTANCE APIs
# =========================
def otsw_between_series_fast(model: OTSWModel, s_ref: int, s_cmp: int) -> float:
    """
    OTSW(s_ref, s_cmp) với cache:
      cost = sum_e w_e * |M[e, s_ref] - M[e, s_cmp]|
    """
    if model.M is None or model.w is None:
        raise ValueError("Model caches not computed. Build the model with build_otsw_tamle().")
    w = model.w
    M = model.M
    diff = np.abs(M[:, s_ref] - M[:, s_cmp])
    return float((w * diff).sum())


def otsw_between_series(model: OTSWModel, s_ref: int, s_cmp: int, p: int = 1) -> float:
    """
    Hiện tại hỗ trợ p=1 (W1 trên cây).
    """
    if p != 1:
        raise ValueError("Currently only supports p=1 (tree-W1).")
    return otsw_between_series_fast(model, s_ref, s_cmp)

In [ ]:

# ============================================================
# ABLATION STUDY FOR OTSW PARAMETERS (k-NN Accuracy & MAP)
# ============================================================
"""
Ablation Study for OTSW Hyperparameters
=======================================

This section performs ablation study on OTSW hyperparameters measuring k-NN performance:
- Lambda (lam_time): (0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100)
- Max Depth: (5, 10, 15, 20, 25, 30)
- Number of Trees: (1, 3, 5, 7, 9, 11, 13, 15)
- Number of Clusters (k_split): (2, 4, 8, 16, 32)

Default values: lambda=5, depth=30, trees=5, num_cluster=2

When varying one parameter, all others are fixed at default values.
Results include Accuracy, MAP, and execution time for each configuration.
"""

import matplotlib.pyplot as plt

# ---------------------- Configuration ----------------------
# Default parameter values
ABLATION_DEFAULT_LAMBDA = 5
ABLATION_DEFAULT_DEPTH = 30
ABLATION_DEFAULT_TREES = 5
ABLATION_DEFAULT_NUM_CLUSTER = 2  # k_split

# Parameter ranges for ablation
ABLATION_LAMBDA_VALUES = [v**2 for v in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100, 1000, 10000]]
# [1e-06, 2.5e-05, 0.0001, 0.0025, 0.01, 0.25, 1, 25, 100, 2500, 10000, 1000000, 100000000]
ABLATION_DEPTH_VALUES = [5, 10, 15, 20, 25, 30]
ABLATION_TREES_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]
ABLATION_NUM_CLUSTER_VALUES = [2, 4, 8, 12, 16, 32, 64, 128, 256, 1024, 4096, 10000] 

# Dataset configuration
ABLATION_DATASET = "BasicMotions"
ABLATION_DATATYPE = "UCR_TSL"
ABLATION_LEAF_SIZE = 16
ABLATION_BASE_SEED = 0
ABLATION_K_NN = 1  # k for k-NN


def run_knn_ablation_single(
    X_train, y_train, X_test, y_test,
    lam_time=ABLATION_DEFAULT_LAMBDA,
    max_depth=ABLATION_DEFAULT_DEPTH,
    num_trees=ABLATION_DEFAULT_TREES,
    k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    leaf_size=ABLATION_LEAF_SIZE,
    seed=ABLATION_BASE_SEED,
    k_nn=ABLATION_K_NN,
):
    """
    Run k-NN with OTSW for a single parameter configuration.
    Returns: (Accuracy, MAP, time_total)
    """
    start_time = time.time()
    
    train_len = len(y_train)
    test_len = len(y_test)
    
    # Build sequences: test first, then train
    sequences = [np.asarray(X_test[i], dtype=float) for i in range(test_len)] + \
                [np.asarray(X_train[j], dtype=float) for j in range(train_len)]
    
    # Build OTSW model with multiple trees and average
    dist_accumulator = np.zeros((test_len, train_len), dtype=float)
    train_dist_accumulator = np.zeros((train_len, train_len), dtype=float)
    
    for t in range(num_trees):
        current_seed = seed + t
        
        model_otsw = build_otsw_tamle(
            sequences,
            lam_time=lam_time,
            leaf_size=leaf_size,
            max_depth=max_depth,
            seed=current_seed,
            k_split=k_split,
        )
        
        M_edge_mass = model_otsw.M  # (E, m_total)
        w = model_otsw.w.reshape(-1, 1)  # (E, 1)
        
        # Test-train distances
        for i in range(test_len):
            dist_all = (w * np.abs(M_edge_mass[:, i:i+1] - M_edge_mass)).sum(axis=0)
            dist_accumulator[i, :] += dist_all[test_len : test_len + train_len]
        
        # Train-train distances
        for i in range(train_len):
            idx_i = test_len + i
            dist_all = (w * np.abs(M_edge_mass[:, idx_i:idx_i+1] - M_edge_mass)).sum(axis=0)
            train_dist_accumulator[i, :] += dist_all[test_len : test_len + train_len]
    
    X_test_computed = dist_accumulator / num_trees
    X_computed = train_dist_accumulator / num_trees
    
    # Run k-NN
    clf = neighbors.KNeighborsClassifier(n_neighbors=k_nn, metric="precomputed")
    clf.fit(X_computed, y_train)
    y_pred = clf.predict(X_test_computed)
    acc = 100.0 * accuracy_score(y_test, y_pred)
    
    # Compute MAP
    map_score = compute_map_knn_precomputed(X_computed, X_test_computed, y_train, y_test, k=k_nn)
    
    time_total = time.time() - start_time
    
    return acc, map_score, time_total


def run_knn_ablation_for_param(
    X_train, y_train, X_test, y_test,
    param_name, param_values, num_runs=5, **fixed_params
):
    """
    Run ablation study for a single parameter with multiple runs.
    Returns: DataFrame with columns [param_value, Accuracy_mean, Accuracy_std, MAP_mean, MAP_std, Time_mean, Time_std]
    """
    results = []
    
    print(f"\n{'='*60}")
    print(f"Ablation Study (k-NN): {param_name}")
    print(f"Testing {len(param_values)} values: {param_values}")
    print(f"Number of runs per value: {num_runs}")
    print(f"Fixed params: {fixed_params}")
    print(f"{'='*60}")
    
    for val in param_values:
        params = fixed_params.copy()
        params[param_name] = val
        
        print(f"  Testing {param_name}={val}...")
        
        acc_runs = []
        map_runs = []
        time_runs = []
        
        for run_idx in range(num_runs):
            try:
                # Use different seed for each run
                params_with_seed = params.copy()
                params_with_seed['seed'] = ABLATION_BASE_SEED + run_idx * 100
                
                acc, map_score, time_total = run_knn_ablation_single(
                    X_train, y_train, X_test, y_test, **params_with_seed
                )
                acc_runs.append(acc)
                map_runs.append(map_score)
                time_runs.append(time_total)
                print(f"    Run {run_idx+1}/{num_runs}: ACC={acc:.2f}%, MAP={map_score:.2f}%, Time={time_total:.2f}s")
            except Exception as e:
                print(f"    Run {run_idx+1}/{num_runs}: ERROR: {e}")
                acc_runs.append(np.nan)
                map_runs.append(np.nan)
                time_runs.append(np.nan)
        
        # Calculate mean and std
        acc_mean = float(np.nanmean(acc_runs))
        acc_std = float(np.nanstd(acc_runs))
        map_mean = float(np.nanmean(map_runs))
        map_std = float(np.nanstd(map_runs))
        time_mean = float(np.nanmean(time_runs))
        time_std = float(np.nanstd(time_runs))
        
        print(f"    => Mean: ACC={acc_mean:.2f}±{acc_std:.2f}%, MAP={map_mean:.2f}±{map_std:.2f}%, Time={time_mean:.2f}±{time_std:.2f}s")
        
        results.append({
            param_name: val,
            "Accuracy_mean": acc_mean,
            "Accuracy_std": acc_std,
            "MAP_mean": map_mean,
            "MAP_std": map_std,
            "Time_mean": time_mean,
            "Time_std": time_std,
        })
    
    return pd.DataFrame(results)


def plot_knn_ablation_results(df, param_name, save_dir="."):
    """
    Plot ablation results: Accuracy, MAP, and Time vs parameter value with error bars (std).
    Saves directly to the specified directory (default: current directory).
    For lam_time, tick labels show sqrt(value) since the code applies sqrt internally.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # For lam_time: display sqrt(value) as label (code uses sqrt(lam_time) internally)
    if param_name == "lam_time":
        x = [f"{v**0.5:.4g}" for v in df[param_name]]
        display_name = "λ"
    else:
        x = df[param_name].astype(str).tolist()
        display_name = param_name
    x_numeric = range(len(x))
    
    # Plot Accuracy with error bars
    axes[0].errorbar(x_numeric, df["Accuracy_mean"], yerr=df["Accuracy_std"], 
                     marker='o', linewidth=2, markersize=8, color='blue', 
                     capsize=4, capthick=1.5, elinewidth=1.5)
    axes[0].set_xlabel(display_name, fontsize=12)
    axes[0].set_ylabel("Accuracy (%)", fontsize=12)
    axes[0].set_title("Accuracy", fontsize=14)
    axes[0].set_xticks(x_numeric)
    axes[0].set_xticklabels(x, rotation=45, ha='right')
    axes[0].grid(True, alpha=0.3)
    
    # Plot MAP with error bars
    axes[1].errorbar(x_numeric, df["MAP_mean"], yerr=df["MAP_std"], 
                     marker='s', linewidth=2, markersize=8, color='green',
                     capsize=4, capthick=1.5, elinewidth=1.5)
    axes[1].set_xlabel(display_name, fontsize=12)
    axes[1].set_ylabel("MAP (%)", fontsize=12)
    axes[1].set_title("MAP", fontsize=14)
    axes[1].set_xticks(x_numeric)
    axes[1].set_xticklabels(x, rotation=45, ha='right')
    axes[1].grid(True, alpha=0.3)
    
    # Plot Time with error bars
    axes[2].errorbar(x_numeric, df["Time_mean"], yerr=df["Time_std"], 
                     marker='^', linewidth=2, markersize=8, color='red',
                     capsize=4, capthick=1.5, elinewidth=1.5)
    axes[2].set_xlabel(display_name, fontsize=12)
    axes[2].set_ylabel("Time (seconds)", fontsize=12)
    axes[2].set_title("Execution Time", fontsize=14)
    axes[2].set_xticks(x_numeric)
    axes[2].set_xticklabels(x, rotation=45, ha='right')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure directly in save_dir
    fig_path = os.path.join(save_dir, f"ablation_knn_{param_name}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    print(f"✅ Plot saved to {fig_path}")
    return fig_path


# Number of runs for ablation study
ABLATION_NUM_RUNS = 5


def run_full_knn_ablation_study(
    dataset_name=ABLATION_DATASET,
    datatype=ABLATION_DATATYPE,
    save_dir=".",
    num_runs=ABLATION_NUM_RUNS,
):
    """
    Run complete ablation study for all OTSW parameters measuring k-NN performance.
    Each parameter configuration is run num_runs times (default: 5) to compute mean and std.
    Results are saved directly in save_dir (default: current directory).
    """
    print(f"\n{'#'*70}")
    print(f"# OTSW ABLATION STUDY (k-NN) ON DATASET: {dataset_name}")
    print(f"{'#'*70}")
    
    # Load dataset
    if datatype == "UCR_TSL":
        X_train, y_train, X_test, y_test = load_ucr_dataset_tsl("../data/UCR", dataset_name)
    elif datatype == "Human_Actions":
        X_train, y_train, X_test, y_test = load_human_action_dataset("../data/Human_Actions", dataset_name)
    else:
        raise ValueError(f"Unknown datatype: {datatype}")
    
    print(f"\nDataset: {dataset_name}")
    print(f"Train samples: {len(y_train)}, Test samples: {len(y_test)}")
    print(f"\nDefault parameters:")
    print(f"  - Lambda (lam_time): {ABLATION_DEFAULT_LAMBDA}")
    print(f"  - Max Depth: {ABLATION_DEFAULT_DEPTH}")
    print(f"  - Number of Trees: {ABLATION_DEFAULT_TREES}")
    print(f"  - Number of Clusters (k_split): {ABLATION_DEFAULT_NUM_CLUSTER}")
    print(f"  - Number of runs per config: {num_runs}")
    
    all_results = {}
    
    '''
    # 1. Ablation on Lambda (lam_time)
    print("\n" + "="*70)
    print("1. ABLATION ON LAMBDA (lam_time)")
    print("="*70)
    df_lambda = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="lam_time",
        param_values=ABLATION_LAMBDA_VALUES,
        num_runs=num_runs,
        max_depth=ABLATION_DEFAULT_DEPTH,
        num_trees=ABLATION_DEFAULT_TREES,
        k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    )
    df_lambda.to_csv(os.path.join(save_dir, "ablation_knn_lambda.csv"), index=False)
    plot_knn_ablation_results(df_lambda, "lam_time", save_dir)
    all_results["lambda"] = df_lambda
    '''
    # 2. Ablation on Max Depth
    print("\n" + "="*70)
    print("2. ABLATION ON MAX DEPTH")
    print("="*70)
    df_depth = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="max_depth",
        param_values=ABLATION_DEPTH_VALUES,
        num_runs=num_runs,
        lam_time=ABLATION_DEFAULT_LAMBDA,
        num_trees=ABLATION_DEFAULT_TREES,
        k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    )
    df_depth.to_csv(os.path.join(save_dir, "ablation_knn_depth.csv"), index=False)
    plot_knn_ablation_results(df_depth, "max_depth", save_dir)
    all_results["depth"] = df_depth
    
    # 3. Ablation on Number of Trees
    print("\n" + "="*70)
    print("3. ABLATION ON NUMBER OF TREES")
    print("="*70)
    df_trees = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="num_trees",
        param_values=ABLATION_TREES_VALUES,
        num_runs=num_runs,
        lam_time=ABLATION_DEFAULT_LAMBDA,
        max_depth=ABLATION_DEFAULT_DEPTH,
        k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    )
    df_trees.to_csv(os.path.join(save_dir, "ablation_knn_trees.csv"), index=False)
    plot_knn_ablation_results(df_trees, "num_trees", save_dir)
    all_results["trees"] = df_trees
    '''
    # 4. Ablation on Number of Clusters (k_split)
    print("\n" + "="*70)
    print("4. ABLATION ON NUMBER OF CLUSTERS (k_split)")
    print("="*70)
    df_cluster = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="k_split",
        param_values=ABLATION_NUM_CLUSTER_VALUES,
        num_runs=num_runs,
        lam_time=ABLATION_DEFAULT_LAMBDA,
        max_depth=ABLATION_DEFAULT_DEPTH,
        num_trees=ABLATION_DEFAULT_TREES,
    )
    df_cluster.to_csv(os.path.join(save_dir, "ablation_knn_cluster.csv"), index=False)
    plot_knn_ablation_results(df_cluster, "k_split", save_dir)
    all_results["cluster"] = df_cluster
    '''
    # Summary
    print("\n" + "#"*70)
    print("# ABLATION STUDY (k-NN) COMPLETE")
    print("#"*70)
    print(f"\nResults saved to {save_dir}:")
    print("  - ablation_knn_lambda.csv + ablation_knn_lam_time.png")
    print("  - ablation_knn_depth.csv + ablation_knn_max_depth.png")
    print("  - ablation_knn_trees.csv + ablation_knn_num_trees.png")
    print("  - ablation_knn_cluster.csv + ablation_knn_k_split.png")
    
    return all_results


# ---------------------- Usage ----------------------
# Run the full ablation study (5 runs per config by default):
all_results = run_full_knn_ablation_study(dataset_name="ItalyPowerDemand", datatype="UCR_TSL", save_dir=".", num_runs=1)

#
# Or run individual parameter ablations:
#   X_train, y_train, X_test, y_test = load_ucr_dataset_tsl("../data/UCR", "BasicMotions")
#   df_lambda = run_knn_ablation_for_param(X_train, y_train, X_test, y_test, "lam_time", ABLATION_LAMBDA_VALUES, num_runs=5, max_depth=30, num_trees=5, k_split=2)
#
# CSV output columns: param_value, Accuracy_mean, Accuracy_std, MAP_mean, MAP_std, Time_mean, Time_std
